In [1]:
# 1️⃣ IMPORTS
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# 2️⃣ FILE PATHS (edit as needed)
clinical_file = r"C:\Users\Neg\TCGA_GC_project\tcga_stad_clinical.tsv.gz"
counts_file = r"C:\Users\Neg\TCGA_GC_project\tcga_stad_counts.tsv.gz"
# You can also use survival file later

# 3️⃣ READ DATA
# Read clinical
clinical = pd.read_csv(clinical_file, sep='\t', index_col=0)

# Read counts
counts = pd.read_csv(counts_file, sep='\t', index_col=0)

# Show info
print("Clinical shape:", clinical.shape)
print("Counts shape:", counts.shape)

# 4️⃣ SELECT YOUR GENE OF INTEREST
# Example: HIST1H3J
gene_of_interest = "HIST1H3J"

# Check if present in counts
matching_genes = [g for g in counts.index if gene_of_interest.upper() in g.upper()]
print("Matching genes:", matching_genes)

# Extract expression for that gene
expr = counts.loc[matching_genes].T  # transpose to samples x gene
expr.columns = ['Expression']

# 5️⃣ JOIN clinical + expression
combined = clinical.copy()
combined['Expression'] = expr['Expression']

# 6️⃣ PLOTTING FUNCTIONS
def plot_box(variable, title, save_path):
    plt.figure(figsize=(8,6))
    sns.boxplot(data=combined, x=variable, y='Expression', palette="Set2")
    sns.stripplot(data=combined, x=variable, y='Expression', color='black', size=3, jitter=True, alpha=0.5)
    plt.title(title)
    plt.ylabel("Transcript per million")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"✅ Saved: {save_path}")

# 7️⃣ PLOTS FOR YOUR PROJECT

# --- 1) Tumor vs Normal ---
if 'sample_type' in combined.columns:
    plot_box('sample_type',
             f"{gene_of_interest} expression in Tumor vs Normal",
             rf"C:\Users\Neg\TCGA_GC_project\Plots\{gene_of_interest}_Tumor_vs_Normal.png")

# --- 2) Tumor Stage ---
stage_cols = [c for c in combined.columns if 'stage' in c.lower()]
print("Stage columns:", stage_cols)

if stage_cols:
    plot_box(stage_cols[0],
             f"{gene_of_interest} expression by Tumor Stage",
             rf"C:\Users\Neg\TCGA_GC_project\Plots\{gene_of_interest}_Tumor_Stage.png")

# --- 3) Tumor Grade ---
grade_cols = [c for c in combined.columns if 'grade' in c.lower()]
print("Grade columns:", grade_cols)

if grade_cols:
    plot_box(grade_cols[0],
             f"{gene_of_interest} expression by Tumor Grade",
             rf"C:\Users\Neg\TCGA_GC_project\Plots\{gene_of_interest}_Tumor_Grade.png")

# --- 4) Nodal Metastasis (N stage) ---
nodal_cols = [c for c in combined.columns if 'nodal' in c.lower()]
print("Nodal columns:", nodal_cols)

if nodal_cols:
    plot_box(nodal_cols[0],
             f"{gene_of_interest} expression by Nodal Metastasis",
             rf"C:\Users\Neg\TCGA_GC_project\Plots\{gene_of_interest}_Nodal_Metastasis.png")

# --- 5) TP53 Mutation Status ---
# If you have a column like 'tp53_mutation_status' or 'tp53'
tp53_cols = [c for c in combined.columns if 'tp53' in c.lower()]
print("TP53 columns:", tp53_cols)

if tp53_cols:
    plot_box(tp53_cols[0],
             f"{gene_of_interest} expression by TP53 Mutation Status",
             rf"C:\Users\Neg\TCGA_GC_project\Plots\{gene_of_interest}_TP53_Mutation.png")

# --- Done ---
print("🎉 All done!")


Clinical shape: (511, 85)
Counts shape: (60660, 448)
Matching genes: []


ValueError: Length mismatch: Expected axis has 0 elements, new values have 1 elements

In [2]:
print(counts.index[:10])


Index(['ENSG00000000003.15', 'ENSG00000000005.6', 'ENSG00000000419.13',
       'ENSG00000000457.14', 'ENSG00000000460.17', 'ENSG00000000938.13',
       'ENSG00000000971.16', 'ENSG00000001036.14', 'ENSG00000001084.13',
       'ENSG00000001167.14'],
      dtype='object', name='Ensembl_ID')


In [3]:
# Strip ENSEMBL version numbers:
counts.index = counts.index.str.split(".").str[0]

# Check again:
print(counts.index[:10])


Index(['ENSG00000000003', 'ENSG00000000005', 'ENSG00000000419',
       'ENSG00000000457', 'ENSG00000000460', 'ENSG00000000938',
       'ENSG00000000971', 'ENSG00000001036', 'ENSG00000001084',
       'ENSG00000001167'],
      dtype='object', name='Ensembl_ID')


In [4]:
from mygene import MyGeneInfo

# Initialize
mg = MyGeneInfo()

# Convert index to list
ensembl_ids = list(counts.index)

# Query MyGeneInfo
print("🔄 Mapping ENSEMBL → SYMBOL...")
res = mg.querymany(ensembl_ids, scopes='ensembl.gene', fields='symbol', species='human')

# Convert to DataFrame
mapped_df = pd.DataFrame(res)

# Drop unmapped
mapped_df = mapped_df.dropna(subset=['symbol'])

# Show mapping
print("✅ Mapped symbols example:")
print(mapped_df[['query', 'symbol']].head())


Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


🔄 Mapping ENSEMBL → SYMBOL...


72 input query terms found dup hits:	[('ENSG00000002586', 2), ('ENSG00000124333', 2), ('ENSG00000124334', 2), ('ENSG00000167393', 2), ('E
1242 input query terms found no hit:	['ENSG00000112096', 'ENSG00000130723', 'ENSG00000131484', 'ENSG00000132832', 'ENSG00000137808', 'ENS


✅ Mapped symbols example:
             query  symbol
0  ENSG00000000003  TSPAN6
1  ENSG00000000005    TNMD
2  ENSG00000000419    DPM1
3  ENSG00000000457   SCYL3
4  ENSG00000000460   FIRRM


In [5]:
# Set index to SYMBOL
symbol_mapping = dict(zip(mapped_df['query'], mapped_df['symbol']))

# Rename index
counts_symbol = counts.rename(index=symbol_mapping)

# Check result
print("✅ Counts with SYMBOL index:")
print(counts_symbol.index[:10])


✅ Counts with SYMBOL index:
Index(['TSPAN6', 'TNMD', 'DPM1', 'SCYL3', 'FIRRM', 'FGR', 'CFH', 'FUCA2',
       'GCLC', 'NFYA'],
      dtype='object', name='Ensembl_ID')


In [8]:
gene_of_interest = "ATP4A"

matching_genes = [g for g in counts_symbol.index if gene_of_interest.upper() == g.upper()]
print("Matching genes:", matching_genes)

if len(matching_genes) == 0:
    print("🚨 Gene not found.")
else:
    # Extract expression
    expr = counts_symbol.loc[matching_genes].T
    expr.columns = ['Expression']

    # Merge with clinical
    combined = clinical.copy()
    combined['Expression'] = expr['Expression']

    print("✅ Combined shape:", combined.shape)
    print(combined.head())


Matching genes: ['ATP4A']
✅ Combined shape: (511, 86)
                                                    id  \
sample                                                   
TCGA-CD-8536-01A  b2f6d8f3-3182-4e5e-be20-8470cfb832c5   
TCGA-R5-A7ZI-01A  b4376da2-05ca-448f-a4bb-e7b62695491a   
TCGA-RD-A7BW-01A  b54fe07a-f63d-4861-878a-8e8322e32fed   
TCGA-BR-8676-01A  b604b620-cd32-4c87-ace3-00a68ab16973   
TCGA-CD-A487-01A  b7d2401e-7d64-4c89-a74c-16d634cafe7f   

                                  disease_type  \
sample                                           
TCGA-CD-8536-01A  Adenomas and Adenocarcinomas   
TCGA-R5-A7ZI-01A  Adenomas and Adenocarcinomas   
TCGA-RD-A7BW-01A  Adenomas and Adenocarcinomas   
TCGA-BR-8676-01A  Adenomas and Adenocarcinomas   
TCGA-CD-A487-01A  Adenomas and Adenocarcinomas   

                                               case_id  submitter_id  \
sample                                                                 
TCGA-CD-8536-01A  b2f6d8f3-3182-4e5e-be20-84

In [10]:
# 📦 Imports
import seaborn as sns
import matplotlib.pyplot as plt
import os

# 📊 FUNCTION to plot and save
def plot_boxplot(combined_df, gene_name, clinical_variable, output_folder):
    # ✅ Check output folder
    os.makedirs(output_folder, exist_ok=True)
    
    # ✅ Plot
    plt.figure(figsize=(8, 6))
    sns.boxplot(data=combined_df, x=clinical_variable, y='Expression', palette='Set2')
    sns.stripplot(data=combined_df, x=clinical_variable, y='Expression', color='black', alpha=0.4, jitter=True)
    
    # ✅ Labels
    plt.title(f"Expression of {gene_name} vs {clinical_variable}", fontsize=14)
    plt.ylabel("Expression (log2 CPM or counts)")  # Adjust depending on your data
    plt.xlabel(clinical_variable)
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    # ✅ Save
    filename = f"{gene_name}_vs_{clinical_variable.replace('.','_')}.png"
    full_path = os.path.join(output_folder, filename)
    plt.savefig(full_path, dpi=300)
    plt.close()
    
    print(f"✅ Saved: {full_path}")


In [11]:
# Set output folder
output_folder = r"C:\Users\Neg\TCGA_GC_project\Expression_analysis_jupyter"

# Example: plot Stage
plot_boxplot(combined, "ATP4A", "ajcc_pathologic_stage.diagnoses", output_folder)

# Example: plot Grade
plot_boxplot(combined, "ATP4A", "tumor_grade.diagnoses", output_folder)

# Example: plot N stage
plot_boxplot(combined, "ATP4A", "ajcc_pathologic_n.diagnoses", output_folder)

# Example: plot Histological Subtype
plot_boxplot(combined, "ATP4A", "histological_type.diagnoses", output_folder)

# Example: Sample type
plot_boxplot(combined, "ATP4A", "tissue_type.samples", output_folder)

# Example: Gender
plot_boxplot(combined, "ATP6A", "gender.demographic", output_folder)

# Example: Race
plot_boxplot(combined, "ATP4A", "race.demographic", output_folder)

# Example: H. pylori (you may need to map this if present!)
# plot_boxplot(combined, "ATP4A", "h_pylori_status.samples", output_folder)  # Example name!


C:\Users\Neg\AppData\Local\Temp\ipykernel_9800\313758947.py:13: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=combined_df, x=clinical_variable, y='Expression', palette='Set2')


✅ Saved: C:\Users\Neg\TCGA_GC_project\Expression_analysis_jupyter\ATP4A_vs_ajcc_pathologic_stage_diagnoses.png


C:\Users\Neg\AppData\Local\Temp\ipykernel_9800\313758947.py:13: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=combined_df, x=clinical_variable, y='Expression', palette='Set2')


✅ Saved: C:\Users\Neg\TCGA_GC_project\Expression_analysis_jupyter\ATP4A_vs_tumor_grade_diagnoses.png


C:\Users\Neg\AppData\Local\Temp\ipykernel_9800\313758947.py:13: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=combined_df, x=clinical_variable, y='Expression', palette='Set2')


✅ Saved: C:\Users\Neg\TCGA_GC_project\Expression_analysis_jupyter\ATP4A_vs_ajcc_pathologic_n_diagnoses.png


ValueError: Could not interpret value `histological_type.diagnoses` for `x`. An entry with this name does not appear in `data`.

<Figure size 800x600 with 0 Axes>

In [12]:
genes = ["ATP4A", "ATP6A"]  # Example list
variables = [
    "ajcc_pathologic_stage.diagnoses",
    "tumor_grade.diagnoses",
    "ajcc_pathologic_n.diagnoses",
    "histological_type.diagnoses",
    "tissue_type.samples",
    "gender.demographic",
    "race.demographic"
]

for gene in genes:
    for var in variables:
        plot_boxplot(combined, gene, var, output_folder)


C:\Users\Neg\AppData\Local\Temp\ipykernel_9800\313758947.py:13: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=combined_df, x=clinical_variable, y='Expression', palette='Set2')


✅ Saved: C:\Users\Neg\TCGA_GC_project\Expression_analysis_jupyter\ATP4A_vs_ajcc_pathologic_stage_diagnoses.png


C:\Users\Neg\AppData\Local\Temp\ipykernel_9800\313758947.py:13: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=combined_df, x=clinical_variable, y='Expression', palette='Set2')


✅ Saved: C:\Users\Neg\TCGA_GC_project\Expression_analysis_jupyter\ATP4A_vs_tumor_grade_diagnoses.png


C:\Users\Neg\AppData\Local\Temp\ipykernel_9800\313758947.py:13: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=combined_df, x=clinical_variable, y='Expression', palette='Set2')


✅ Saved: C:\Users\Neg\TCGA_GC_project\Expression_analysis_jupyter\ATP4A_vs_ajcc_pathologic_n_diagnoses.png


ValueError: Could not interpret value `histological_type.diagnoses` for `x`. An entry with this name does not appear in `data`.

<Figure size 800x600 with 0 Axes>

In [13]:
print("Columns in combined:")
print(combined.columns.tolist())


Columns in combined:
['id', 'disease_type', 'case_id', 'submitter_id', 'primary_site', 'alcohol_history.exposures', 'race.demographic', 'gender.demographic', 'ethnicity.demographic', 'vital_status.demographic', 'age_at_index.demographic', 'days_to_birth.demographic', 'year_of_birth.demographic', 'year_of_death.demographic', 'primary_site.project', 'project_id.project', 'disease_type.project', 'name.project', 'name.program.project', 'tissue_source_site_id.tissue_source_site', 'code.tissue_source_site', 'name.tissue_source_site', 'project.tissue_source_site', 'bcr_id.tissue_source_site', 'days_to_death.demographic', 'entity_submitter_id.annotations', 'notes.annotations', 'submitter_id.annotations', 'classification.annotations', 'entity_id.annotations', 'created_datetime.annotations', 'annotation_id.annotations', 'entity_type.annotations', 'updated_datetime.annotations', 'case_id.annotations', 'state.annotations', 'category.annotations', 'status.annotations', 'case_submitter_id.annotation

In [14]:
"ajcc_pathologic_stage.diagnoses"


'ajcc_pathologic_stage.diagnoses'

In [15]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import scipy.stats as stats
import statannotations  # If not installed: pip install statannotations

# Your combined dataframe is already named: combined
# Output folder
output_folder = r"C:\Users\Neg\TCGA_GC_project\Expression_analysis_jupyter"
os.makedirs(output_folder, exist_ok=True)

# Function to plot boxplot with p-values
def plot_boxplot_with_pvalues(combined_df, gene_name, clinical_variable, output_folder):
    print(f"\n📊 Plotting: {clinical_variable}")

    # Drop missing values for this variable
    df_plot = combined_df[[clinical_variable, 'Expression']].dropna()

    # Sort categories (optional)
    if df_plot[clinical_variable].dtype == "object":
        categories = sorted(df_plot[clinical_variable].unique())
    else:
        categories = df_plot[clinical_variable].unique()

    # Create the plot
    plt.figure(figsize=(8, 6))
    ax = sns.boxplot(data=df_plot, x=clinical_variable, y='Expression', palette='Set2')
    sns.stripplot(data=df_plot, x=clinical_variable, y='Expression', color='black', alpha=0.4, jitter=True)

    # Add p-value automatically (Kruskal-Wallis for multiple groups)
    groups = [df_plot[df_plot[clinical_variable] == cat]['Expression'] for cat in categories]
    if len(groups) > 1:
        stat, p = stats.kruskal(*groups)
        p_text = f"Kruskal-Wallis p = {p:.3e}"
        plt.title(f"{gene_name} expression by {clinical_variable}\n{p_text}", fontsize=14)
    else:
        plt.title(f"{gene_name} expression by {clinical_variable}", fontsize=14)

    plt.ylabel("Expression (log2 scale)", fontsize=12)
    plt.xlabel(clinical_variable, fontsize=12)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()

    # Save the figure
    save_path = os.path.join(output_folder, f"{gene_name}_by_{clinical_variable.replace('.', '_')}.png")
    plt.savefig(save_path, dpi=300)
    plt.close()

    print(f"✅ Saved to: {save_path}")

# ---- List of variables you can plot ----

variables_to_plot = [
    "ajcc_pathologic_stage.diagnoses",
    "tumor_grade.diagnoses",
    "primary_diagnosis.diagnoses",
    "gender.demographic",
    "race.demographic",
    "ajcc_pathologic_n.diagnoses",
    "sample_type.samples",
]

# ---- Run for each variable ----

for var in variables_to_plot:
    plot_boxplot_with_pvalues(combined, "ATP6A", var, output_folder)

print("\n🎉 All plots done!")



📊 Plotting: ajcc_pathologic_stage.diagnoses


C:\Users\Neg\AppData\Local\Temp\ipykernel_9800\1052764922.py:28: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.boxplot(data=df_plot, x=clinical_variable, y='Expression', palette='Set2')


✅ Saved to: C:\Users\Neg\TCGA_GC_project\Expression_analysis_jupyter\ATP6A_by_ajcc_pathologic_stage_diagnoses.png

📊 Plotting: tumor_grade.diagnoses


C:\Users\Neg\AppData\Local\Temp\ipykernel_9800\1052764922.py:28: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.boxplot(data=df_plot, x=clinical_variable, y='Expression', palette='Set2')


✅ Saved to: C:\Users\Neg\TCGA_GC_project\Expression_analysis_jupyter\ATP6A_by_tumor_grade_diagnoses.png

📊 Plotting: primary_diagnosis.diagnoses


...

✅ Saved to: C:\Users\Neg\TCGA_GC_project\Expression_analysis_jupyter\ATP6A_by_race_demographic.png

📊 Plotting: ajcc_pathologic_n.diagnoses


C:\Users\Neg\AppData\Local\Temp\ipykernel_9800\1052764922.py:28: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.boxplot(data=df_plot, x=clinical_variable, y='Expression', palette='Set2')


✅ Saved to: C:\Users\Neg\TCGA_GC_project\Expression_analysis_jupyter\ATP6A_by_ajcc_pathologic_n_diagnoses.png

📊 Plotting: sample_type.samples


C:\Users\Neg\AppData\Local\Temp\ipykernel_9800\1052764922.py:28: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.boxplot(data=df_plot, x=clinical_variable, y='Expression', palette='Set2')


✅ Saved to: C:\Users\Neg\TCGA_GC_project\Expression_analysis_jupyter\ATP6A_by_sample_type_samples.png

🎉 All plots done!


In [5]:
!pip install statannotations
